# A clean TensorFlow CNN training pipeline

**Learning objective:** Use `tf.data`, batching, prefetching and explicit validation/test contracts.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:15:57.221462: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974957.238952    4018 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974957.243395    4018 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:15:58.980418: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
(x,y),(xt,yt)=tf.keras.datasets.fashion_mnist.load_data(); x=x[:6000,...,None].astype("float32")/255; y=y[:6000]
xtr,xva=x[:5000],x[5000:]; ytr,yva=y[:5000],y[5000:]
train_ds=tf.data.Dataset.from_tensor_slices((xtr,ytr)).shuffle(5000,seed=SEED).batch(64).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices((xva,yva)).batch(128).prefetch(tf.data.AUTOTUNE)
batch=next(iter(train_ds)); print("one batch images:",batch[0].shape,"labels:",batch[1].shape); print("train batches:",tf.data.experimental.cardinality(train_ds).numpy())


one batch images: (64, 28, 28, 1) labels: (64,)
train batches: 79


In [3]:
model=tf.keras.Sequential([tf.keras.layers.Input((28,28,1)),tf.keras.layers.Conv2D(16,3,padding="same",activation="relu"),tf.keras.layers.MaxPooling2D(),tf.keras.layers.GlobalAveragePooling2D(),tf.keras.layers.Dense(10,activation="softmax")])
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
h=model.fit(train_ds,validation_data=val_ds,epochs=1,verbose=0)
display(pd.DataFrame(h.history).round(4))


,accuracy,loss,val_accuracy,val_loss
0,0.101,2.2876,0.113,2.2756


`tf.data` separates data movement from model definition. In larger systems it becomes the place for parsing, caching, parallel mapping, shuffling and prefetching.
